In [6]:
import os, io, base64, numpy as np, requests, torch, torch.nn as nn

BASE_URL= "http://154.57.164.80:35779"
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

# ---------- codecs ----------
def x01_from_b64_png(b64):
    from PIL import Image
    img = Image.open(io.BytesIO(base64.b64decode(b64))).convert("L")
    if img.size != (28, 28): raise ValueError("Expected 28x28 PNG")
    return np.clip(np.asarray(img, dtype=np.float32) / 255.0, 0.0, 1.0)

def b64_png_from_x01(x2d):
    from PIL import Image
    x255 = np.clip((x2d * 255.0).round(), 0, 255).astype(np.uint8)
    buf = io.BytesIO(); Image.fromarray(x255, mode="L").save(buf, format="PNG", optimize=True)
    return base64.b64encode(buf.getvalue()).decode("ascii")

def l2(a, b):
    return float(np.linalg.norm((a - b).ravel()))

# ---------- model ----------
class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1); self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25); self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128); self.fc2 = nn.Linear(128, 10)
    def forward(self, x01):
        x = (x01 - MNIST_MEAN) / MNIST_STD
        x = torch.relu(self.conv1(x)); x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2); x = self.dropout1(x)
        x = torch.flatten(x, 1); x = torch.relu(self.fc1(x))
        x = self.dropout2(x); x = self.fc2(x)
        return torch.log_softmax(x, dim=1)

# ---------- fetch ----------
ch  = requests.get(f"{BASE_URL}/challenge", timeout=10).json()
x   = x01_from_b64_png(ch["image_b64"])
lab = int(ch["label"]); tgt = int(ch["target"]); thr = float(ch["l2_threshold"])
print("label:", lab, "target:", tgt, "l2_threshold:", thr)

open("deepfool_weights.pth","wb").write(requests.get(f"{BASE_URL}/weights", timeout=10).content)
model = SimpleClassifier().eval()
model.load_state_dict(torch.load("deepfool_weights.pth", map_location="cpu"))
print("local clean pred:", int(model(torch.from_numpy(x[None,None]).float()).argmax(1)))

# ---------- targeted L2 PGD ----------
# Minimize CE toward target; after each step, if we exceed the budget,
# project the perturbation back onto the L2 ball (leave headroom for PNG rounding).
def attack(x_np, target, budget, steps=1000, step_size=0.03):
    x0  = torch.from_numpy(x_np[None, None]).float()
    tgt = int(target)
    delta = torch.zeros_like(x0, requires_grad=True)
    for i in range(steps):
        logits = model(x0 + delta)[0]            # log-softmax outputs (monotonic in logits)
        t = logits[tgt]
        other = logits.clone(); other[tgt] = -1e9
        margin = other.max() - t                 # <0 once target is the top class
        grad = torch.autograd.grad(margin, delta)[0]
        with torch.no_grad():
            g = grad / (grad.view(-1).norm() + 1e-12)   # unit L2 direction
            delta -= step_size * g                       # minimize the margin
            d = (x0 + delta).clamp(0, 1) - x0            # respect pixel range
            n = d.view(-1).norm()
            if n > budget:                               # project onto L2 ball
                d = d * (budget / n)
            delta.copy_(d)
        delta.requires_grad_(True)
    return (x0 + delta).clamp(0, 1).detach().squeeze().numpy().astype(np.float32)

# use nearly the full budget; verify against the PNG round-trip
best = None
for frac in [0.95, 0.97, 0.98, 0.93, 0.90]:
    adv = attack(x, tgt, budget=thr * frac, steps=1200, step_size=0.03)
    b64 = b64_png_from_x01(adv)
    adv_rt = x01_from_b64_png(b64)
    pred = int(model(torch.from_numpy(adv_rt[None, None]).float()).argmax(1))
    d = l2(adv_rt, x)
    print(f"frac={frac:.2f}  pred={pred} (need {tgt})  l2(after PNG)={d:.4f} (<= {thr})")
    if pred == tgt and d <= thr:
        best = b64; print("  -> OK, submitting"); break

if best:
    r = requests.post(f"{BASE_URL}/submit", json={"image_b64": best}, timeout=10)
    print(r.status_code, r.text)
else:
    print("no candidate met both conditions — see note below")

# ---------- submit ----------
r = requests.post(f"{BASE_URL}/submit", json={"image_b64": b64}, timeout=10)
print(r.status_code, r.text)

label: 4 target: 6 l2_threshold: 0.75
local clean pred: 4
frac=0.95  pred=6 (need 6)  l2(after PNG)=0.7140 (<= 0.75)
  -> OK, submitting
200 {"ok":true,"flag":"[REDACTED]","pred":6,"target":6,"l2":0.7139947414398193}
200 {"ok":true,"flag":"[REDACTED]","pred":6,"target":6,"l2":0.7139947414398193}
